In [1]:
import hashlib
import numpy as np
import pandas as pd
# Load the dataset
df = pd.read_csv('./data/customer_support_raw.csv')

In [2]:
row = df.iloc[12]
print(f"Customer info:\n {row[['conversation_id', 'customer_id', 'customer_name', 'email', 'phone', 'customer_type', 'zipcode', 'timestamp',]]}\n")
print(f"Instruction: {row['instruction_raw'][:150]} \n")
print(f"Response: {row['response_raw'][:150]} ...\n")

Customer info:
 conversation_id                 conv_000012
customer_id                      cust_64305
customer_name                  Dan Martinez
email              dan.martinez@outlook.com
phone                          356-441-3187
customer_type                    enterprise
zipcode                                2101
timestamp               2025-01-02 10:42:58
Name: 12, dtype: object

Instruction: problem with canceling purchase ORD730143  You can reach me at 356-441-3187. 

Response: I'm aware that you're facing difficulties canceling your purchase with the order number ORD769583. I apologize for any inconvenience this may have cau ...



In [3]:
print(f"Starting with {len(df)} conversations")

# Track issues found
issues = {
    'missing_timestamp': 0,
    'invalid_customer_id': 0,
    'near_empty_message': 0
}

# Remove missing timestamps
missing_timestamp = df[df['timestamp'].isna()] # Rows with missing timestamps
issues['missing_timestamp'] = len(missing_timestamp)
df = df.drop(missing_timestamp.index)

# Remove invalid customer IDs (must start with 'cust_')
invalid_id = df[~df['customer_id'].str.startswith('cust_')] # Customer IDs not starting with 'cust_' are considered invalid
issues['invalid_customer_id'] = len(invalid_id)
df = df.drop(invalid_id.index)

# Remove empty messages
near_empty_message = df[df['instruction_raw'].str.strip().str.len() < 3] # Messages with less than 3 characters are considered near empty
issues['near_empty_message'] = len(near_empty_message)
df = df.drop(near_empty_message.index)

# Print validation report
print("\n=== Validation Report ===")
print(f"Removed {issues['missing_timestamp']} rows with missing timestamps")
print(f"Removed {issues['invalid_customer_id']} rows with invalid customer IDs")
print(f"Removed {issues['near_empty_message']} rows with near empty messages")
print(f"Clean dataset: {len(df)} conversations remaining")

Starting with 26872 conversations

=== Validation Report ===
Removed 134 rows with missing timestamps
Removed 160 rows with invalid customer IDs
Removed 28 rows with near empty messages
Clean dataset: 26550 conversations remaining


In [4]:
import re
EMAIL_PATTERN = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
PHONE_PATTERN = r'\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b'

# Function to remove personally identifiable information (PII) from text
def remove_pii(text):
    if pd.isna(text):
        return text
    text = re.sub(EMAIL_PATTERN, '[EMAIL]', text) #remove email pattern
    text =  re.sub(PHONE_PATTERN, '[PHONE]', text) #remove phone pattern pattern
    return text

df['instruction_cleaned'] = df['instruction_raw'].apply(remove_pii)
df['response_cleaned'] = df['response_raw'].apply(remove_pii)

print("\n=== Before/After Examples ===")
samples = df[df['instruction_raw'].str.contains(EMAIL_PATTERN, regex=True, na=False) | df['instruction_raw'].str.contains(PHONE_PATTERN, regex=True, na=False)].head(2)
for _, row in samples.iterrows():
    print(f"Before: {row['instruction_raw'][:100]}")
    print(f"After:  {row['instruction_cleaned'][:100]}\n")


=== Before/After Examples ===
Before: problem with canceling purchase ORD730143  You can reach me at 356-441-3187.
After:  problem with canceling purchase ORD730143  You can reach me at [PHONE].

Before: can you help me canceling order ORD883802?  My email is bobl@email.com.
After:  can you help me canceling order ORD883802?  My email is [EMAIL].



In [ ]:
# Anonymize customer IDs by hashing them and drop other PII columns

def hash_customer_id(customer_id):
    if pd.isna(customer_id):
        return None
    return hashlib.sha256(customer_id.encode()).hexdigest() if customer_id else None

sample_id = df['customer_id'].iloc[0]
print("=== Anonymization Example ===")
hashed_sample_id = hash_customer_id(sample_id)
print(f"Original: {sample_id} -> Hashed: {hashed_sample_id[:16] if hashed_sample_id else 'N/A'}...")

df['customer_id_hashed'] = df['customer_id'].apply(hash_customer_id)
df = df.drop(columns=['customer_id', 'customer_name', 'email', 'phone'])
print("Dropped: customer_id, customer_name, email, phone")

=== Anonymization Example ===
Original: cust_60150 -> Hashed: 2085cd119c93815c...
Dropped: customer_id, customer_name, email, phone


In [7]:
zipcode_counts = df['zipcode'].value_counts()
noise = np.random.laplace(0, 5.0, len(zipcode_counts))
private_counts = (zipcode_counts + noise).clip(0).astype(int)

print("\n=== Differential Privacy ===")
print(f"{'Zipcode':<10} {'Original':>10} {'Private':>10}")
for zipcode in zipcode_counts.index[:5]:
    print(f"{zipcode:<10} {zipcode_counts[zipcode]:>10} {private_counts[zipcode]:>10.0f}")

at_risk = zipcode_counts[zipcode_counts < 5]
if len(at_risk) > 0:
    print(f"\n⚠️  Zipcodes with <5 conversations protected: {list(at_risk.index)}")


=== Differential Privacy ===
Zipcode      Original    Private
10002            1341       1345
60601            1312       1297
90002            1307       1333
94106            1285       1275
10003            1284       1281

⚠️  Zipcodes with <5 conversations protected: [32202, 85001, 28202, 92101, 78701]
